# 4. Statistics

Two-sample Kolmogorov-Smirnov tests between the clusters of notebook 2, with a
Holm correction over each family of comparisons.  Nothing is written out; the
tables are printed.

| input | what it provides |
| --- | --- |
| `results/umap_5100_6450_dim2_nei12_eps0475.csv` | `params.csv` with `umap_x`, `umap_y` and `DB_label` appended, the `n_neighbors = 12` run of notebook 2 |


## Imports


In [10]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

## Configuration


In [ ]:
# Input location: the notebook 2 run the paper quotes
RESULTS_PATH = './results/umap_5100_6450_dim2_nei12_eps0475.csv'

# Cluster labels of that run, as notebook 3 names them
LABEL_COLUMN = 'DB_label'
HV_LABEL = 1
NV1_LABEL = 3          # together with SN_type == 'NV'
T_LABEL = 3            # together with SN_type == '91T'
NV2_LABEL = 5
BG_LABEL = 4

# The decline-rate test keeps only the supernovae that have a measured Dm15
MAX_DECLINE_RATE = 3


In [4]:
data = pd.read_csv(RESULTS_PATH)


In [20]:
# data_pick = data[data['D15'] < MAX_DECLINE_RATE]
data_pick = data
NV1 = data_pick[(data_pick[LABEL_COLUMN] == NV1_LABEL) & (data_pick['SN_type'] == 'NV')]
T = data_pick[(data_pick[LABEL_COLUMN] == T_LABEL) & (data_pick['SN_type'] == '91T')]
NV2 = data_pick[data_pick[LABEL_COLUMN] == NV2_LABEL]
NV = pd.concat([NV1, NV2])
HV = data_pick[data_pick[LABEL_COLUMN] == HV_LABEL]
bg = data_pick[data_pick[LABEL_COLUMN] == BG_LABEL]

In [21]:
def ks_test_with_holm_Rsn(groups, column):
    """
    Two-sample K-S tests of `column` between the groups listed below.

    Returns the K-S statistic D, the raw p-value and the Holm-adjusted p-value.
    """

    comparisons = [
        ('HV vs NV1',  groups['HV'],  groups['NV1']),
        ('HV vs NV2',  groups['HV'],  groups['NV2']),
        ('NV1 vs NV2', groups['NV1'], groups['NV2']),
    ]

    results = []

    for name, group1, group2 in comparisons:
        # drop NaN
        sample1 = group1[column].dropna()
        sample2 = group2[column].dropna()

        ks_result = stats.ks_2samp(sample1, sample2)

        results.append({
            'Comparison': name,
            'N1': len(sample1),
            'N2': len(sample2),
            'D': ks_result.statistic,
            'p_raw': ks_result.pvalue,
        })

    result_df = pd.DataFrame(results)

    # Holm correction over the comparisons above as one family
    reject, p_holm, _, _ = multipletests(
        result_df['p_raw'],
        alpha=0.05,
        method='holm'
    )

    result_df['p_Holm'] = p_holm
    result_df['Significant'] = reject

    return result_df

In [22]:
def ks_test_with_holm_others(groups, column):
    """
    Two-sample K-S tests of `column` between the groups listed below.

    Returns the K-S statistic D, the raw p-value and the Holm-adjusted p-value.
    """

    comparisons = [
        ('HV vs NV1',  groups['HV'],  groups['NV1']),
        ('HV vs NV2',  groups['HV'],  groups['NV2']),
        ('NV1 vs NV2', groups['NV1'], groups['NV2']),
        ('HV vs NV',   groups['HV'],  groups['NV']),
    ]

    results = []

    for name, group1, group2 in comparisons:
        # drop NaN
        sample1 = group1[column].dropna()
        sample2 = group2[column].dropna()

        ks_result = stats.ks_2samp(sample1, sample2)

        results.append({
            'Comparison': name,
            'N1': len(sample1),
            'N2': len(sample2),
            'D': ks_result.statistic,
            'p_raw': ks_result.pvalue,
        })

    result_df = pd.DataFrame(results)

    # Holm correction over the comparisons above as one family
    reject, p_holm, _, _ = multipletests(
        result_df['p_raw'],
        alpha=0.05,
        method='holm'
    )

    result_df['p_Holm'] = p_holm
    result_df['Significant'] = reject

    return result_df

In [23]:
groups_Rsn = {
    'HV': HV,
    'NV1': NV1,
    'NV2': NV2,
}

groups_others = {
    'HV': HV,
    'NV1': NV1,
    'NV2': NV2,
    'NV': NV,
}

ratio_result = ks_test_with_holm_Rsn(groups_Rsn, 'ratio')
Rsi_result   = ks_test_with_holm_Rsn(groups_Rsn, 'R_si')

In [24]:
print('=== R_sn / R_gal ===')
print(
    ratio_result.to_string(
        index=False,
        formatters={
            'D': '{:.4f}'.format,
            'p_raw': '{:.4g}'.format,
            'p_Holm': '{:.4g}'.format,
        }
    )
)

print('\n=== R(Si II) ===')
print(
    Rsi_result.to_string(
        index=False,
        formatters={
            'D': '{:.4f}'.format,
            'p_raw': '{:.4g}'.format,
            'p_Holm': '{:.4g}'.format,
        }
    )
)

=== R_sn / R_gal ===
Comparison  N1  N2      D     p_raw   p_Holm  Significant
 HV vs NV1  28  24 0.1429    0.9141   0.9141        False
 HV vs NV2  28  33 0.4686  0.001469 0.002937         True
NV1 vs NV2  24  33 0.5303 0.0004061 0.001218         True

=== R(Si II) ===
Comparison  N1  N2      D     p_raw   p_Holm  Significant
 HV vs NV1  28  24 0.2798    0.2159   0.2159        False
 HV vs NV2  28  33 0.3755   0.01983  0.03966         True
NV1 vs NV2  24  33 0.5341 0.0003522 0.001057         True


In [25]:
def ks_test_with_holm_Rsn_mc(groups, ratio_col='ratio', err_col='ratio_err',
                              n_trials=10000, random_state=None):
    """
    K-S test on `ratio_col`, perturbed by its measurement error `err_col`.

    Each trial redraws every ratio from a Gaussian of that width and repeats the
    K-S test and the Holm correction; `n_trials` trials in all.  Returns the
    median and the 16-84 percentile (1 sigma) of D, p_raw and p_Holm.
    """
    rng = np.random.default_rng(random_state)

    comparisons = [
        ('HV vs NV1',  'HV',  'NV1'),
        ('HV vs NV2',  'HV',  'NV2'),
        ('NV1 vs NV2', 'NV1', 'NV2'),
    ]

    # Take ratio and ratio_err out once: NaN dropped, so the sample size is
    # the same in every trial
    values, errors, n_sample = {}, {}, {}
    for key, df in groups.items():
        sub = df[[ratio_col, err_col]].dropna()
        values[key] = sub[ratio_col].to_numpy()
        errors[key] = sub[err_col].to_numpy()
        n_sample[key] = len(sub)

    D_trials = {name: np.empty(n_trials) for name, _, _ in comparisons}
    p_raw_trials = {name: np.empty(n_trials) for name, _, _ in comparisons}
    p_holm_trials = {name: np.empty(n_trials) for name, _, _ in comparisons}

    for i in range(n_trials):
        # Perturb each group once and reuse it across the three comparisons,
        # so that the Holm family is the same in every trial
        perturbed = {key: rng.normal(values[key], errors[key]) for key in values}

        trial_D, trial_p = [], []
        for name, k1, k2 in comparisons:
            ks_result = stats.ks_2samp(perturbed[k1], perturbed[k2])
            trial_D.append(ks_result.statistic)
            trial_p.append(ks_result.pvalue)

        _, p_holm, _, _ = multipletests(trial_p, alpha=0.05, method='holm')

        for j, (name, _, _) in enumerate(comparisons):
            D_trials[name][i] = trial_D[j]
            p_raw_trials[name][i] = trial_p[j]
            p_holm_trials[name][i] = p_holm[j]

    rows = []
    for name, k1, k2 in comparisons:
        D_med, D_lo, D_hi = np.percentile(D_trials[name], [50, 16, 84])
        p_raw_med, p_raw_lo, p_raw_hi = np.percentile(p_raw_trials[name], [50, 16, 84])
        p_holm_med, p_holm_lo, p_holm_hi = np.percentile(p_holm_trials[name], [50, 16, 84])

        rows.append({
            'Comparison': name,
            'N1': n_sample[k1],
            'N2': n_sample[k2],
            'D': D_med,
            'p_raw': p_raw_med,
            'p_Holm': p_holm_med,
            'p_Holm_lo': p_holm_lo,
            'p_Holm_hi': p_holm_hi,
            'Significance': 'Yes' if p_holm_med < 0.05 else 'No',
        })

    return pd.DataFrame(rows)

groups_ratio_mc = {
    'HV': HV,
    'NV1': NV1,
    'NV2': NV2,
}

mc_result = ks_test_with_holm_Rsn_mc(
    groups_ratio_mc,
    ratio_col='ratio',
    err_col='ratio_err',
    n_trials=10000,
    random_state=0,
)

print('=== R_sn / R_gal (measurement-error Monte Carlo, N=10000) ===')
for _, row in mc_result.iterrows():
    print(
        f"{row['Comparison']:>11}  N1={row['N1']:>3}  N2={row['N2']:>3}  "
        f"D={row['D']:.4f}  p_raw={row['p_raw']:.4g}  "
        f"p_Holm={row['p_Holm']:.4g} "
        f"({row['p_Holm_lo']:.4g}–{row['p_Holm_hi']:.4g})  "
        f"{row['Significance']}"
    )

=== R_sn / R_gal (measurement-error Monte Carlo, N=10000) ===
  HV vs NV1  N1= 28  N2= 24  D=0.1726  p_raw=0.7724  p_Holm=0.7724 (0.556–0.9141)  No
  HV vs NV2  N1= 28  N2= 33  D=0.4470  p_raw=0.002911  p_Holm=0.006252 (0.001651–0.02278)  Yes
 NV1 vs NV2  N1= 24  N2= 33  D=0.4697  p_raw=0.002623  p_Holm=0.007237 (0.00213–0.02172)  Yes


In [19]:
data_pick = data[data['D15'] < MAX_DECLINE_RATE]
NV1 = data_pick[(data_pick[LABEL_COLUMN] == NV1_LABEL) & (data_pick['SN_type'] == 'NV')]
T = data_pick[(data_pick[LABEL_COLUMN] == T_LABEL) & (data_pick['SN_type'] == '91T')]
NV2 = data_pick[data_pick[LABEL_COLUMN] == NV2_LABEL]
NV = pd.concat([NV1, NV2])
HV = data_pick[data_pick[LABEL_COLUMN] == HV_LABEL]
bg = data_pick[data_pick[LABEL_COLUMN] == BG_LABEL]

groups_d15 = {
    'HV': HV,
    'NV1': NV1,
    'NV2': NV2,
}

D15_result = ks_test_with_holm_Rsn(groups_d15, 'D15')

print('\n=== Delta m_15 ===')
print(
    D15_result.to_string(
        index=False,
        formatters={
            'D': '{:.4f}'.format,
            'p_raw': '{:.4g}'.format,
            'p_Holm': '{:.4g}'.format,
        }
    )
)


=== Delta m_15 ===
Comparison  N1  N2      D    p_raw    p_Holm  Significant
 HV vs NV1  25  22 0.4927 0.004051  0.008102         True
 HV vs NV2  25  31 0.2529   0.2785    0.2785        False
NV1 vs NV2  22  31 0.5601 0.000294 0.0008819         True
